In [ ]:
# !pip install transformers torch accelerate
# !pip install dotenv


   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 2/2 [dotenv]



### Text generation in the Pipeline function

In [ ]:
from transformers import pipeline
from dotenv import load_dotenv
import os
load_dotenv()  # Load environment variables from .env file
hf_token = os.getenv("hugging_face_token")  # Access the Hugging Face token from environment variables
generator = pipeline(
    "text-generation",
    model="gpt2",  # small and fast
    token=hf_token  # Pass the Hugging Face token
)

output = generator(
    """Question: What is the weather in bangalore in india look like?
Answer:""",
    max_new_tokens=50,
    do_sample=True,          # ✅ important
    temperature=0.7,         # ✅ controls randomness
    top_k=50,                # ✅ improves diversity
    pad_token_id=50256       # ✅ removes warning
)

print(output[0]["generated_text"])

# print(output[0]["generated_text"])

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2825.10it/s]
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the weather in bangalore in india look like?
Answer: In Bangalore, the weather is very good and there is no rain.
In Bangalore, the weather is very good and there is no rain. What is the weather in bhutan in india look like?
Answer: The weather is very


### without pipeline - the acutal code

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name).to(device)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

prompt = """
Question: What is the weather in bangalore in india look like?
Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.1,
    top_p=0.9,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(output[0], skip_special_tokens=True)) 

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1872.02it/s]



Question: What is the weather in bangalore in india look like?
Answer:
A lot of people are looking for a place to live. The city has been getting quite hot lately, so it's not surprising that there have been some bad days and good nights here too! I've seen many places where you can get away from your home by walking around with friends or going out on dates (I'm sure they're all pretty cool). But if we were lucky enough at any


GPT-2 has no knowledge of current events, real-time data, or your location. It was trained on internet text up to 2019 and simply predicts likely next tokens. When you ask "What is the weather in Bangalore?" it generates statistically plausible-sounding weather text — not real data.

This is exactly why RAG exists. RAG solves this by retrieving real, current information first, then passing it to the model as context. The model's job then becomes: "given this retrieved context, generate an accurate answer" — not "hallucinate from training memory."

In [1]:
from transformers import pipeline
classifier = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")

complaints = [
  "The fresh produce section was completely empty on Sunday",
  "Staff were extremely helpful when I couldn't find the product",
  "Self-checkout machines kept breaking down during busy hours"
]

results = classifier(complaints)
for text, result in zip(complaints, results):
  print(f"{result['label']} ({result['score']:.2f}): {text[:50]}")

d:\LLM-Learning-Journey\LLM-Learning-Journey\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\LLM-Learning-Journey\LLM-Learning-Journey\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\srini\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to ru

NEGATIVE (1.00): The fresh produce section was completely empty on 
POSITIVE (0.99): Staff were extremely helpful when I couldn't find 
NEGATIVE (1.00): Self-checkout machines kept breaking down during b


In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = """
Question: What is the weather in bangalore in india look like?
Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2811.88it/s]


In [12]:
input_ids = inputs.input_ids

In [13]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [14]:
model_output = model(input_ids)

In [15]:
model_output

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[ -39.8041,  -36.4310,  -39.6659,  ...,  -50.0735,  -50.2993,
           -38.4815],
         [ -84.3976,  -85.3610,  -85.2695,  ...,  -94.7732,  -93.4589,
           -83.8213],
         [-135.3676, -134.3287, -135.9463,  ..., -142.6832, -145.4111,
          -131.1902],
         ...,
         [ -43.8989,  -44.9641,  -46.0630,  ...,  -55.5446,  -55.5348,
           -45.4459],
         [-118.7037, -119.4748, -120.7343,  ..., -127.4730, -124.9274,
          -115.6707],
         [ -80.8351,  -74.2813,  -76.6284,  ...,  -92.0208,  -90.9631,
           -81.8208]]], grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)

In [16]:
lm_head_output = model.lm_head(model_output[0])

RuntimeError: mat1 and mat2 shapes cannot be multiplied (20x50257 and 768x50257)